# Project: Unicorn Industry Analysis (2019–2021)

**Objective:** Analyze unicorn company creation between 2019 and 2021 to identify the three industries that produced the highest number of new unicorns and compare their yearly growth and average company valuations.

**Business Question:** Which industries produced the most new unicorn companies between 2019 and 2021, and how did their yearly unicorn counts and average valuations compare?

**Tables Used:**

* `dates`

  * `company_id` : Unique company identifier
  * `date_joined` : Date the company achieved unicorn status
  * `year_founded` : Year the company was founded

* `industries`

  * `company_id` : Unique company identifier
  * `industry` : Industry classification

* `funding`

  * `company_id` : Unique company identifier
  * `valuation` : Company valuation (USD)

**Methodology:**

1. Analyze unicorn companies that achieved unicorn status between 2019 and 2021 and identify the three industries with the highest number of new unicorns.
2. Combine company industry, unicorn year, and valuation information across the relevant datasets.
3. Group the results by industry and year to examine changes in unicorn creation over time.
4. Calculate the number of new unicorns and average company valuation for each industry-year combination.
5. Convert average valuations to billions of USD and organize the results by year and unicorn count.

**Output Columns:**

* `industry` : Industry name
* `year` : Year the company became a unicorn
* `num_unicorns` : Number of new unicorns created that year
* `avg_valuation_billions` : Average company valuation in billions of USD


In [1]:
-- Identifying the three best-performing industries based on the number of unicorns created in 2019-2021.

WITH top_3 AS (
	SELECT 
		i.industry,
		COUNT(i.*) AS num_unicorns
	FROM dates AS d
	RIGHT JOIN industries AS i
	ON i.company_id = d.company_id
	LEFT JOIN companies AS c
	ON c.company_id = i.company_id
	WHERE DATE_PART('year', d.date_joined) IN (2019, 2020, 2021)
	GROUP BY i.industry
	ORDER BY num_unicorns DESC
	LIMIT 3
	),

-- Joining top_3 CTE with the industry and dates tables to only keep the top 3 best-performing industries while acquiring the year joined for each company_id.
	
unicorn_year AS (
	SELECT
		i.industry,
		i.company_id,
		DATE_PART('year', d.date_joined) AS year
	FROM industries AS i
	INNER JOIN top_3 AS t
	ON i.industry = t.industry
	INNER JOIN dates AS d
	ON d.company_id = i.company_id
	WHERE DATE_PART('year', d.date_joined) IN (2019, 2020, 2021)
	),

-- Joining the unicorn_year CTE with the funding table to get the valuation for each company_id. Also, converting the valuations to billions.
	
valuations AS (
	SELECT 
		u.*, 
		f.valuation/1000000000 AS valuation_billions
	FROM unicorn_year AS u
	LEFT JOIN funding AS f
	ON u.company_id = f.company_id
	)

-- Calculating the average valuation and the number of unicorns for the top 3 industries in each year (2019, 2020, 2021).
	
SELECT 
	industry, 
	year, 
	COUNT(company_id) AS num_unicorns,
	ROUND(AVG(valuation_billions), 2) AS average_valuation_billions
FROM valuations
GROUP BY industry, year
ORDER BY year DESC, num_unicorns DESC

,industry,year,num_unicorns,average_valuation_billions
0,Fintech,2021,138,2.75
1,Internet software & services,2021,119,2.15
2,E-commerce & direct-to-consumer,2021,47,2.47
3,Internet software & services,2020,20,4.35
4,E-commerce & direct-to-consumer,2020,16,4.00
5,Fintech,2020,15,4.33
6,Fintech,2019,20,6.80
7,Internet software & services,2019,13,4.23
8,E-commerce & direct-to-consumer,2019,12,2.58
